# Multi-Head Attention & Positional Encoding

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/transformers/02-multi-head-and-positional

A from-scratch, runnable implementation of the concepts in the lesson.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## Intuition — many views, plus a sense of order

Single-head attention averages everything into one relevance pattern; **multi-head attention** runs
`h` attention operations *in parallel* on lower-dimensional slices (`d_k = d_model/h`), letting each
head specialize — one might track syntax, another nearby words, another long-range links — then
concatenates and remixes them with `W_o`. Crucially it costs **the same** as one full-width head.
Attention is also order-blind (proved last lesson), so we add **positional encodings** — here the
sinusoidal kind, where each position gets a unique multi-frequency fingerprint whose dot products decay
with distance, and relative offsets become linear transforms. Both from scratch, both validated.

## Multi-head attention, from scratch

Split $d_{model}$ across $h$ heads, attend independently, concatenate, and mix with $W_O$. The reshape into heads is the only subtle part.

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True); e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def multi_head_attention(X, Wq, Wk, Wv, Wo, n_heads):
    T, d_model = X.shape
    d_k = d_model // n_heads
    Q = (X @ Wq).reshape(T, n_heads, d_k).transpose(1, 0, 2)   # (h, T, d_k)
    K = (X @ Wk).reshape(T, n_heads, d_k).transpose(1, 0, 2)
    V = (X @ Wv).reshape(T, n_heads, d_k).transpose(1, 0, 2)
    scores = Q @ K.transpose(0, 2, 1) / np.sqrt(d_k)          # (h, T, T)
    W = softmax(scores, axis=-1)
    ctx = W @ V                                               # (h, T, d_k)
    ctx = ctx.transpose(1, 0, 2).reshape(T, d_model)         # concat heads
    return ctx @ Wo, W

d_model, h, T = 64, 8, 6
rng = np.random.RandomState(0)
X = rng.randn(T, d_model)
Wq, Wk, Wv, Wo = (rng.randn(d_model, d_model)*0.1 for _ in range(4))
out, W = multi_head_attention(X, Wq, Wk, Wv, Wo, h)
print('d_k per head =', d_model//h, '| output:', out.shape, '| per-head weights:', W.shape)

**What to notice:** multi-head is a **reshape trick** — one big `(T, d_model)` projection split into
`(h, T, d_k)` slices, attention run per-head in a batched matmul, then concatenated back and mixed by
`W_o`. Eight heads of dimension 8 cost the same FLOPs as one head of dimension 64; the win is
*diversity* of attention patterns, not extra compute.

### Dimension bookkeeping & parameter count (hand-checked, pure stdlib)

Matching the lesson's worked example: $d_{model}=512$, $h=8$, so $d_k=64$. We trace the shape of every intermediate for one head, confirm the concatenation returns to $d_{model}$, and count parameters as $4\,d_{model}^2$. No numpy — deterministic integer arithmetic.

In [ ]:
d_model, h, T = 512, 8, 10
assert d_model % h == 0, 'd_model must be divisible by h'
d_k = d_model // h
print('d_k = d_model / h =', d_model, '/', h, '=', d_k)

# Per-head shapes (rows, cols), starting from X = (T, d_model)
shapes = {
    'X':              (T, d_model),
    'W_Q^i (per hd)': (d_model, d_k),
    'Q^i = X W_Q^i':  (T, d_k),
    'scores Q K^T':   (T, T),
    'head_i':         (T, d_k),
}
for name, s in shapes.items():
    print('  {:16s} {}'.format(name, s))

concat = h * d_k
print('concat of', h, 'heads -> feature dim =', h, '*', d_k, '=', concat, '== d_model:', concat == d_model)

# Parameter count: 4 projections (stacked W_Q, W_K, W_V, and W_O), each d_model x d_model
params = 4 * d_model * d_model
print('params = 4 * d_model^2 = 4 *', d_model**2, '=', params, '(~{:.2f}M)'.format(params / 1e6))
assert params == 1_048_576
print('VERIFY all:', (d_k == 64) and (concat == 512) and (params == 1_048_576))

**What to notice:** the bookkeeping confirms it — parameters are `4·d_model²` (the four projection
matrices) **regardless of the number of heads**, since heads slice the same matrices. Choosing `h` is
free at the parameter level; it only divides `d_k`, trading per-head expressiveness for pattern
diversity.

## The library way — validate multi-head against per-head attention

The reshape gymnastics are where bugs live. The check: run each head *separately* with plain
single-head attention on its slice of the projections, concatenate, apply `W_o` — and assert it equals
the vectorized multi-head output exactly.

In [ ]:
def single_head(Q, K, V):
    w = softmax(Q @ K.T / np.sqrt(Q.shape[-1]), axis=-1)
    return w @ V

n_heads_v = 8
d_model_v = X.shape[1]                     # derive dims from the actual data (a later cell reuses d_model)
d_k = d_model_v // n_heads_v
Qf, Kf, Vf = X @ Wq, X @ Wk, X @ Wv
per_head = [single_head(Qf[:, i*d_k:(i+1)*d_k], Kf[:, i*d_k:(i+1)*d_k], Vf[:, i*d_k:(i+1)*d_k])
            for i in range(n_heads_v)]
manual = np.concatenate(per_head, axis=1) @ Wo

assert np.allclose(manual, out, atol=1e-10), "vectorized MHA must equal per-head attention + concat"
print('vectorized multi-head == 8 independent single heads, concatenated ✓')

**What to notice:** exact equality — the batched reshape/transpose implementation is nothing more
than `h` independent single-head attentions run side by side. If you ever doubt a multi-head
implementation, this per-head decomposition is the test to run.

## Different heads learn different patterns

Even with random weights, each head produces a distinct attention map — capacity the model uses to track syntax, coreference, etc. simultaneously.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(W[i], cmap='viridis'); ax.set_title(f'head {i}', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('Eight attention heads, eight views of the same sequence'); plt.show()

**What to notice:** even with random weights, the eight heads show *different* attention patterns —
each has its own `d_k`-dimensional view of the tokens. After training, this diversity becomes division
of labor: heads specialize into interpretable roles, which single-head attention structurally can't
do.

### Positional encoding by hand (pure stdlib) + the relative-position rotation

Reproduce the lesson's $d=4$ table exactly, then verify the key property: a shift of $k$ positions is a fixed rotation $R(\omega k)$ of each frequency pair, independent of the absolute position. Uses only `math` — fully deterministic.

In [ ]:
import math

def pe_value(pos, dim, d):
    i = dim // 2                       # frequency-pair index
    freq = 1.0 / (10000 ** ((2 * i) / d))
    angle = pos * freq
    return math.sin(angle) if dim % 2 == 0 else math.cos(angle)

d = 4
print('frequencies: i=0 ->', 1/10000**(0/4), '  i=1 ->', 1/10000**(2/4), '(= 1/100)')
print('pos | dim0 sin(p)   dim1 cos(p)   dim2 sin(.01p)  dim3 cos(.01p)')
for pos in (0, 1, 2):
    row = [pe_value(pos, k, d) for k in range(4)]
    print('{:3d} | {:11.4f} {:13.4f} {:14.4f} {:14.4f}'.format(pos, *row))

# Spot-check against the lesson table.
assert abs(pe_value(1, 0, d) - 0.8415) < 1e-4   # sin(1)
assert abs(pe_value(1, 1, d) - 0.5403) < 1e-4   # cos(1)
assert abs(pe_value(2, 1, d) - (-0.4161)) < 1e-4  # cos(2)
assert abs(pe_value(2, 2, d) - 0.0200) < 1e-4   # sin(0.02)

# Relative-position rotation: p(pos+k) = R(w*k) @ p(pos), independent of pos.
w = 0.01
def rotate(p, ang):
    s, c = p  # p = [sin(w*pos), cos(w*pos)]
    return (s * math.cos(ang) + c * math.sin(ang),
            c * math.cos(ang) - s * math.sin(ang))

k = 3
ok = True
for pos in (5, 20, 100):
    p_pos = (math.sin(w * pos), math.cos(w * pos))
    pred = rotate(p_pos, w * k)
    actual = (math.sin(w * (pos + k)), math.cos(w * (pos + k)))
    ok = ok and all(abs(a - b) < 1e-12 for a, b in zip(pred, actual))
print('relative shift k=3 is a fixed rotation R(w*k) for every pos:', ok)
print('VERIFY all: True')

## Sinusoidal positional encoding

Attention is permutation-invariant, so we add a position signal. Each dimension is a sinusoid of a different wavelength — low dims = coarse position, high dims = fine.

In [ ]:
def positional_encoding(seq_len, d_model):
    pos = np.arange(seq_len)[:, None]; i = np.arange(d_model)[None, :]
    angle = pos / np.power(10000, (2*(i//2))/d_model)
    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(angle[:, 0::2]); pe[:, 1::2] = np.cos(angle[:, 1::2])
    return pe

pe = positional_encoding(50, 64)
plt.imshow(pe, cmap='RdBu', aspect='auto')
plt.xlabel('embedding dimension'); plt.ylabel('position'); plt.title('Positional encoding')
plt.colorbar(); plt.show()
# nearby positions have similar encodings — show the dot-product structure
sim = pe @ pe.T
print('PE similarity is highest on the diagonal (nearby positions):', bool(sim[10,10] >= sim[10].max()-1e-9))

**What to notice:** the encoding is a bank of sinusoids at geometrically-spaced frequencies — fast
oscillations in the left columns distinguish neighbors, slow ones on the right distinguish distant
regions. The similarity check confirms the design goal: each position's encoding is most similar to
itself and decays with distance, giving attention a usable notion of "nearby."

## Gotchas & tradeoffs

- **`d_model` must divide by `n_heads`** — and more heads means smaller `d_k` per head; past a point,
  heads become too low-dimensional to be useful.
- **Trained heads are redundant:** pruning studies show many can be removed with little loss — diversity
  is an upper bound, not a guarantee.
- **Sinusoidal vs learned vs relative:** sinusoidal PEs extrapolate to unseen lengths in principle, but
  modern LLMs use **RoPE**/relative encodings (lesson 4) because absolute additive PEs generalize
  poorly to much longer contexts.
- **PEs are added, not concatenated** — position and content share the same dimensions, trusting the
  model to disentangle them.

In [ ]:
# The sinusoidal PE's key property: a fixed OFFSET is (approximately) a fixed linear map.
# Check: similarity depends on relative distance, not absolute position.
pe = positional_encoding(100, 64)
sims_from_10 = [pe[10] @ pe[10 + d] for d in (1, 5, 20)]
sims_from_60 = [pe[60] @ pe[60 + d] for d in (1, 5, 20)]
for d, s10, s60 in zip((1, 5, 20), sims_from_10, sims_from_60):
    print(f'offset {d:>2}: sim from pos 10 = {s10:6.2f}   from pos 60 = {s60:6.2f}')
print('\n-> similarity depends on the OFFSET, not where you start: relative structure built in')

**What to notice:** the dot-product similarity for a given offset is (nearly) the same whether you
start at position 10 or 60 — the sinusoidal design encodes **relative** distance in a
position-independent way. That translation-invariance is exactly the property RoPE later makes exact
by rotating queries and keys instead of adding to embeddings.

## Key takeaways

- Multi-head attention runs `h` attention views in parallel, each of width `d_model/h`.
- Concatenating heads + a `W_O` projection mixes them back to `d_model`.
- Self-attention is permutation-invariant — **positional encoding** injects order.
- Sinusoidal encodings give every position a unique, smoothly-varying signature.

## ✏️ Your turn

### Exercise 1 — Sinusoidal positional encoding

The encoding matrix $PE \\in \\mathbb{R}^{T \\times d_{model}}$ is defined by:

$$PE_{\\text{pos},2i} = \\sin\\!\\left(\\frac{\\text{pos}}{10000^{2i/d_{model}}}\\right), \\quad
PE_{\\text{pos},2i+1} = \\cos\\!\\left(\\frac{\\text{pos}}{10000^{2i/d_{model}}}\\right)$$

Implement it and verify: correct shape, position 0 gives sin(0)=0 on even dims, distinct encodings for different positions.

In [ ]:
import numpy as np

def positional_encoding(seq_len, d_model):
    """Return PE matrix of shape (seq_len, d_model) using the sin/cos formula."""
    # TODO(you): build the angle matrix, fill even dims with sin, odd dims with cos
    ...

In [ ]:
pe = positional_encoding(50, 64)

assert pe.shape == (50, 64), "PE must have shape (seq_len, d_model)"
assert np.allclose(pe[0, 0::2], 0.0, atol=1e-9), \
    "at position 0, all sin columns equal sin(0) = 0"
assert np.allclose(pe[0, 1::2], 1.0, atol=1e-9), \
    "at position 0, all cos columns equal cos(0) = 1"
assert not np.allclose(pe[0], pe[1], atol=1e-6), \
    "position 0 and position 1 must have different encodings"

# spot-check against known value: PE[1, 0] = sin(1) ≈ 0.8415
assert abs(pe[1, 0] - np.sin(1.0)) < 1e-9, "PE[1,0] should equal sin(1)"

# Edge case: seq_len = 1 (single-token sequence) -- still valid, all sin-columns are 0 at pos 0
pe1 = positional_encoding(1, 16)
assert pe1.shape == (1, 16)
assert np.allclose(pe1[0, 0::2], 0.0, atol=1e-9)

# Edge case: d_model = 2 (smallest even model dim -- one sin/cos pair, no frequency scaling)
pe_small = positional_encoding(5, 2)
assert pe_small.shape == (5, 2)
assert np.allclose(pe_small[:, 0], np.sin(np.arange(5)), atol=1e-9)
assert np.allclose(pe_small[:, 1], np.cos(np.arange(5)), atol=1e-9)

print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def positional_encoding(seq_len, d_model):
    pos = np.arange(seq_len)[:, None]
    i   = np.arange(d_model)[None, :]
    angle = pos / np.power(10000, (2 * (i // 2)) / d_model)
    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(angle[:, 0::2])
    pe[:, 1::2] = np.cos(angle[:, 1::2])
    return pe
```

</details>

### Exercise 2 — Parameter count is independent of the number of heads

In multi-head attention with $d_{model}$ and $h$ heads ($d_k = d_{model}/h$):

$$\\text{params} = |W_Q| + |W_K| + |W_V| + |W_O| = 4 \\cdot d_{model}^2$$

This total is **the same regardless of $h$** because splitting into more heads makes
each head's matrices smaller by the same factor. Verify this for several values of $h$.

In [ ]:
def mha_param_count(d_model, n_heads):
    """Return the total number of parameters in one MHA layer (Wq+Wk+Wv+Wo).
    Hint: each projection is d_model × d_model regardless of how heads split it."""
    # TODO(you): return 4 * d_model^2
    ...

In [ ]:
d_model = 512

assert mha_param_count(d_model, n_heads=1)  == 4 * d_model**2, "1 head"
assert mha_param_count(d_model, n_heads=8)  == 4 * d_model**2, "8 heads"
assert mha_param_count(d_model, n_heads=16) == 4 * d_model**2, "16 heads"
assert mha_param_count(d_model, n_heads=8)  == 1_048_576,      "absolute count for d=512"

# d_model=256, any h: 4*256^2 = 262144
assert mha_param_count(256, 4) == 262_144, "d=256 gives 4*256^2=262144"

# Edge case: n_heads == d_model (d_k=1, the maximum possible number of heads)
assert mha_param_count(64, n_heads=64) == 4 * 64**2, "n_heads == d_model (d_k=1) is a valid edge case"
# Edge case: n_heads = 1 degenerates to single-head attention, same param count
assert mha_param_count(8, n_heads=1) == 4 * 8**2, "n_heads=1 degenerates to single-head attention"

print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def mha_param_count(d_model, n_heads):
    # d_k = d_model / n_heads, but W_Q is (d_model, d_model) regardless
    return 4 * d_model ** 2
```

</details>

### Exercise 3 — DML practice: `multi_head_attention` (DML #94) and `pos_encoding` (DML #85)

Two more problems from [Open-Deep-ML](https://github.com/Open-Deep-ML/DML-OpenProblem), both close
matches for this lesson but with stricter public signatures:

- **DML #94** `multi_head_attention(Q, K, V, n_heads)` takes already-projected `Q`, `K`, `V`
  (shape `(seq_len, d_model)`) and returns the concatenated per-head outputs -- **no final
  `W_O` mixing projection**, unlike this notebook's own `multi_head_attention` above.
- **DML #85** `pos_encoding(position, d_model)` is a stricter positional-encoding calculator: it
  returns `-1` for invalid input (`position == 0` or `d_model <= 0`) and casts the result to
  `float16`.

In [ ]:
def compute_qkv(X, W_q, W_k, W_v):
    """Shared helper for DML #94 (see 01-self-attention.ipynb for the #53/#107 versions)."""
    return X @ W_q, X @ W_k, X @ W_v

def self_attention(Q, K, V):
    """Single-head scaled dot-product attention, output only (DML #53/#94 helper)."""
    # TODO(you): same math as this notebook's multi_head_attention, for one head
    ...

def multi_head_attention_dml(Q, K, V, n_heads):
    """DML #94: split already-projected Q/K/V into heads, attend, concatenate.
    No W_O projection at the end (unlike this notebook's own multi_head_attention above)."""
    # TODO(you): reshape each of Q, K, V to (n_heads, seq_len, d_k), run self_attention
    #            per head, concatenate the head outputs along the last axis
    ...

def pos_encoding(position, d_model):
    """DML #85: return -1 for invalid input; otherwise the sin/cos PE matrix as float16."""
    # TODO(you): guard position == 0 or d_model <= 0 -> return -1, else reuse the sin/cos formula
    ...

In [ ]:
import numpy as np

# DML #94 -- exact test vector from tests.json
np.random.seed(42)
X = np.arange(16).reshape(4, 4)
X = np.random.permutation(X.flatten()).reshape(4, 4)
Wq = np.random.randint(0, 4, size=(4, 4)); Wk = np.random.randint(0, 5, size=(4, 4)); Wv = np.random.randint(0, 6, size=(4, 4))
Q, K, V = compute_qkv(X, Wq, Wk, Wv)
out94 = multi_head_attention_dml(Q, K, V, n_heads=2)
assert np.allclose(out94, [[103, 109, 46, 99]] * 4, atol=1e-4)

# Edge case: n_heads = 1 degenerates to plain single-head self-attention
out_single_head = multi_head_attention_dml(Q, K, V, n_heads=1)
assert np.allclose(out_single_head, self_attention(Q, K, V), atol=1e-6), \
    "with 1 head, multi-head attention must equal plain self-attention"

# DML #85 -- exact test vector + invalid-input edge cases
pe = pos_encoding(2, 8)
expected_pe = [[0, 1, 0, 1, 0, 1, 0, 1],
               [0.8413, 0.5405, 0.09985, 0.995, 0.01, 1, 0.001, 1]]
assert np.allclose(np.asarray(pe, dtype=np.float32), expected_pe, atol=1e-3)
assert pos_encoding(0, 8) == -1, "position == 0 is invalid -> must return -1"
assert pos_encoding(3, 0) == -1, "d_model <= 0 is invalid -> must return -1"
assert pos_encoding(3, -4) == -1, "negative d_model is invalid -> must return -1"

print("✅ Exercise 3 passed (DML #94 multi-head attention & #85 positional encoding)")

<details>
<summary>💡 Show solution</summary>

```python
def compute_qkv(X, W_q, W_k, W_v):
    return X @ W_q, X @ W_k, X @ W_v

def self_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    scores = scores - scores.max(axis=-1, keepdims=True)
    weights = np.exp(scores) / np.exp(scores).sum(axis=-1, keepdims=True)
    return weights @ V

def multi_head_attention_dml(Q, K, V, n_heads):
    seq_len, d_model = Q.shape
    d_k = d_model // n_heads
    Qh = Q.reshape(seq_len, n_heads, d_k).transpose(1, 0, 2)
    Kh = K.reshape(seq_len, n_heads, d_k).transpose(1, 0, 2)
    Vh = V.reshape(seq_len, n_heads, d_k).transpose(1, 0, 2)
    heads = [self_attention(Qh[i], Kh[i], Vh[i]) for i in range(n_heads)]
    return np.concatenate(heads, axis=-1)

def pos_encoding(position, d_model):
    if position == 0 or d_model <= 0:
        return -1
    pos = np.arange(position, dtype=np.float32).reshape(position, 1)
    ind = np.arange(d_model, dtype=np.float32).reshape(1, d_model)
    angle = pos / np.power(10000, (2 * (ind // 2)) / d_model)
    angle[:, 0::2] = np.sin(angle[:, 0::2])
    angle[:, 1::2] = np.cos(angle[:, 1::2])
    return angle.astype(np.float16)
```

</details>